In [74]:
# !pip install num2words
# !pip install av
# !pip install --upgrade transformers
# !pip install opencv-python

In [75]:
import requests
import os
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import random
import json
import re
from datetime import datetime, timezone
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [76]:
VIDEO_IDS = [25, 26, 27, 28]
NUM_FRAMES = 12
OUTPUT_DIR = Path("..", "Moment-Retrieval-and-Video-Summarization", "Descriptions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TVSUM_DIR = Path("..", "Moment-Retrieval-and-Video-Summarization", "TVSum")

PROMPTS = {
    "short":  "Describe what is happening in the video.",
    "medium": "Describe the key events and actions taking place in the video, providing a concise summary of the main activities and interactions. Focus on visually distinct moments.",
    "long":   """Describe what is happening in the video, sequentially focusing on the most important events.
Describe the events in detail, including the people involved, their appearance, their actions, and any relevant context.
Avoid generic descriptions and focus on specific moments that stand out.
List the events in order of occurrence."""
}

In [77]:
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
).to("cuda")

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.98s/it]


In [78]:
def run_vlm(video_path, prompt_text, num_frames):
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "path": str(video_path)},
            {"type": "text",  "text": prompt_text}
        ]
    }]
    inputs = processor.apply_chat_template(
        messages,
        num_frames=num_frames,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device, dtype=torch.bfloat16)

    generated_ids = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=512,
        repetition_penalty=1.3,
        no_repeat_ngram_size=4,
    )
    output = processor.batch_decode(generated_ids, skip_special_tokens=True)
    response = output[0].split("Assistant:")[-1].strip()

    return {
        "response": response,
        "num_frames": num_frames,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

In [79]:
def save_video_results(video_id, video_path, results):
    out_file = OUTPUT_DIR / f"video_{video_id}.json"
    data = {
        "video_id": video_id,
        "video_path": str(video_path),
        "processed_at": datetime.now(timezone.utc).isoformat(),
        "vlm_results": results
    }
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved:{out_file}")

def add_vlm_result_to_json(video_id, prompt_type, prompt_text, result):
    """Append a single vlm result to the video's JSON file. Creates the file if needed."""
    out_file = OUTPUT_DIR / f"video_{video_id}.json"

    if out_file.exists():
        with open(out_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = {
            "video_id": video_id,
            "vlm_results": []
        }

    data["vlm_results"].append({
        "prompt_type":  prompt_type,
        "prompt":       prompt_text,
        "num_frames":   result["num_frames"],
        "timestamp":    result["timestamp"],
        "response":     result["response"],
    })

    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"    Updated: {out_file}")

In [80]:
for video_id in VIDEO_IDS:
    video_path = TVSUM_DIR / f"video_{video_id}.mp4"

    if not video_path.exists():
        print(f"Problem: video_{video_id}.mp4 not found, skipping")
        continue

    print(f"\nProcessing video_{video_id}...")

    # load existing results for this video if any
    out_file = OUTPUT_DIR / f"video_{video_id}.json"
    if out_file.exists():
        with open(out_file, "r", encoding="utf-8") as f:
            existing = json.load(f)
        already_done = {
            (r["prompt_type"], r["num_frames"]) for r in existing["vlm_results"]
        }
    else:
        already_done = set()

    for prompt_type, prompt_text in PROMPTS.items():
        if (prompt_type, NUM_FRAMES) in already_done:
            print(f"        Skipping '{prompt_type}' at {NUM_FRAMES} frames — already done")
            continue

        print(f"        Running '{prompt_type}' prompt with {NUM_FRAMES} frames...")
        result = run_vlm(video_path, prompt_text, num_frames=NUM_FRAMES)
        add_vlm_result_to_json(video_id, prompt_type, prompt_text, result)
        print(f"        Done — {len(result['response'].split())} words generated")

print("\n\nAll videos processed!")


Processing video_25...
        Running 'short' prompt with 12 frames...
    Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 124 words generated
        Running 'medium' prompt with 12 frames...
    Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 81 words generated
        Running 'long' prompt with 12 frames...
    Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_25.json
        Done — 244 words generated

Processing video_26...
        Running 'short' prompt with 12 frames...
    Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_26.json
        Done — 155 words generated
        Running 'medium' prompt with 12 frames...
    Updated: ..\Moment-Retrieval-and-Video-Summarization\Descriptions\video_26.json
        Done — 93 words generated
        Running 'long' prompt with 12 frames...
    Updated: ..\Moment-Retrieval-and-Video-Summarization\Descript